# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys


# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

from typing import List
from dotenv import load_dotenv

import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

# Verify lib folder exists
if not Path("lib").exists():
    print("WARNING: 'lib' folder not found in", Path().resolve())
    print("Contents:", os.listdir("."))

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import BaseMessage, UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG

import json
import chromadb
from chromadb.utils import embedding_functions
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
load_dotenv()

True

In [4]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")

# Add project root to path so we can import src/
sys.path.insert(0, str(Path("../").resolve()))

openai_client = os.getenv("CHROMA_OPENAI_API_KEY")
chroma_client = chromadb.PersistentClient(path="./chromadb")


db = VectorStoreManager(os.getenv("CHROMA_OPENAI_API_KEY"))

loader_service = CorpusLoaderService(db)

In [5]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OpenAI_API_KEY")
)



In [6]:
rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)


games_market_rag = RAG(
    llm=rag_llm,
    vector_store = loader_service.load_pdf(
        store_name="games_market",
        pdf_path="TheGamingIndustry2024.pdf",
    )
)


VectorStore `games_market` ready!


Pages from `TheGamingIndustry2024.pdf` added!


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [7]:
# retrieve_games_collection tool
# Loads the JSON files from the /games folder into a persistent Chroma collection
# and exposes a semantic-search tool over them.

from pathlib import Path

chroma_client = chromadb.PersistentClient(path="chromadb")

games_embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY")
)

games_collection = chroma_client.get_or_create_collection(
    name="udaplay_games",
    embedding_function=games_embedding_fn,
)

# Load every JSON record in /games into the collection (idempotent: file stem == id)
games_dir = Path("games")
existing_ids = set(games_collection.get()["ids"])

documents, ids, metadatas = [], [], []
for game_file in sorted(games_dir.glob("*.json")):
    game_id = game_file.stem
    if game_id in existing_ids:
        continue

    with open(game_file, "r") as f:
        game = json.load(f)

    document_text = (
        f"Name: {game['Name']}\n"
        f"Platform: {game['Platform']}\n"
        f"Genre: {game['Genre']}\n"
        f"Publisher: {game['Publisher']}\n"
        f"YearOfRelease: {game['YearOfRelease']}\n"
        f"Description: {game['Description']}"
    )

    documents.append(document_text)
    ids.append(game_id)
    metadatas.append({
        "Name": game["Name"],
        "Platform": game["Platform"],
        "Genre": game["Genre"],
        "Publisher": game["Publisher"],
        "YearOfRelease": game["YearOfRelease"],
    })

if documents:
    games_collection.add(documents=documents, ids=ids, metadatas=metadatas)
    print(f"Indexed {len(documents)} games into `udaplay_games`.")
else:
    print(f"`udaplay_games` already contains {len(existing_ids)} games — nothing new to index.")


@tool
def retrieve_games_collection(query: str):
    """
    Semantic search: Finds most relevant games in the vector DB.
    args:
    - query: a question about game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    result = games_collection.query(query_texts=[query], n_results=5)

    hits = []
    for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
        hits.append({
            "Name": meta.get("Name"),
            "Platform": meta.get("Platform"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": doc,
        })
    return hits


`udaplay_games` already contains 15 games — nothing new to index.


In [8]:
from pydantic import BaseModel
#from openai import OpenAI

from openai import OpenAI as OpenAIClient

openai_client = OpenAIClient(api_key=os.getenv("OPENAI_API_KEY"))

class EvaluationReport(BaseModel):
    useful: bool
    description: str

#### Evaluate Retrieval Tool

In [9]:
# evaluate_retrieval tool
# Uses an LLM as judge to decide whether the retrieved documents are sufficient
# to answer the user's question, parsed into an EvaluationReport.

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> EvaluationReport:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    docs_text = "\n\n".join(
        f"Document {i + 1}:\n{doc}" for i, doc in enumerate(retrieved_docs)
    )

    judge_prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query:\n{question}\n\n"
        f"Retrieved Documents:\n{docs_text}"
    )

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are an expert judge of retrieval quality for RAG systems.",
            },
            {"role": "user", "content": judge_prompt},
        ],
        response_format=EvaluationReport,
    )

    return response.choices[0].message.parsed


In [10]:
from tavily import TavilyClient
import os

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

#### Game Web Search Tool

In [11]:
# game_web_search tool — uses the Tavily client to search the web.

@tool
def game_web_search(question: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - question: a question about game industry.
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        topic="general",
        max_results=5,
    )

    results = []
    for result in response.get("results", []):
        results.append({
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content"),
            "score": result.get("score"),
        })

    return results


# Quick sanity check
question = "What are the most popular video games in 2025?"
results = game_web_search(question)

for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"  Title: {result['title']}")
    print(f"  URL: {result['url']}")
    print(f"  Score: {result['score']}")
    print(f"  Content: {result['content'][:200]}...")
    print()


Result 1:
  Title: Ars Technica's Top 20 video games of 2025
  URL: https://arstechnica.com/gaming/2025/12/ars-technicas-top-20-video-games-of-2025
  Score: 0.9365236
  Content: # Ars Technica’s Top 20 video games of 2025

A mix of expected sequels and out-of-nowhere indie gems made 2025 a joy.

When we put together our top 20 games of last year, we specifically called out Ci...

Result 2:
  Title: The most-played video games of 2025 for Polygon's writers and editors
  URL: https://www.polygon.com/most-played-games-2025
  Score: 0.933534
  Content: Polygon.com logo

Sign in now

Follow

Followed

Link copied to clipboard

Add us on

4

By  Polygon Staff

Best of the Year

They're not our GOTYs, but also they totally are our GOTYs

# The games we...

Result 3:
  Title: U.S. best selling games 2025 - Statista
  URL: https://www.statista.com/statistics/1285658/top-ranked-video-games-sales-annual
  Score: 0.921929
  Content: Expert resources to inform and inspire.

## Best-selling video ga

### Agent

In [12]:
# Agent abstraction built on the project's StateMachine (lib.agents.Agent).
# - Equipped with gpt-4o-mini (used elsewhere in the notebook for RAG/judging).
# - Instructions describe the RAG-first → evaluate → web-fallback workflow.
# - All three tools developed above are plugged in.

GAME_AGENT_INSTRUCTIONS = """You are an expert game industry analyst with deep knowledge
of video games, gaming history, platforms, publishers, and market trends.

Answer questions about the game industry accurately, citing the source of your facts.

You have access to these tools:
1. retrieve_games_collection(query): Semantic search over the internal games vector DB.
   ALWAYS try this FIRST for any question about a specific game (release year, platform,
   publisher, description).
2. evaluate_retrieval(question, retrieved_docs): LLM-as-judge. After calling
   retrieve_games_collection, pass the returned items (as strings) into this tool to
   decide whether the documents are sufficient to answer the user's question.
3. game_web_search(question): Tavily-backed web search. Only call this if
   evaluate_retrieval reports the internal documents are NOT useful.

Workflow:
- retrieve_games_collection → evaluate_retrieval → (web search only if needed) → answer.
- Be transparent about which tool produced the information.
- Provide concise, well-structured answers."""


gaming_industry_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=GAME_AGENT_INSTRUCTIONS,
    tools=[retrieve_games_collection, evaluate_retrieval, game_web_search],
    temperature=0.3,
)


def extract_final_text(run: Run) -> str:
    """Pull the last assistant message text out of an Agent Run."""
    final_state = run.get_final_state()
    if not final_state:
        return ""
    for message in reversed(final_state["messages"]):
        if isinstance(message, AIMessage) and message.content:
            return message.content
    return ""


# Sanity check
print("=" * 60)
print("Gaming Industry Agent — sanity check")
print("=" * 60)
sanity_query = "When was Gran Turismo released and on what platform?"
print(f"\nQuery: {sanity_query}\n")
sanity_run = gaming_industry_agent.invoke(sanity_query, session_id="sanity")
print(f"Response:\n{extract_final_text(sanity_run)}")


Gaming Industry Agent — sanity check

Query: When was Gran Turismo released and on what platform?

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Response:
The original **Gran Turismo** was released in **1997** on the **PlayStation 1**. It is known for being a realistic racing simulator that set a new standard for the genre. 

Additionally, **Gran Turismo 5** was released later in **2010** on the **PlayStation 3**, but the original game is the one that established the franchise.


In [13]:
# Invoke the StateMachine-backed agent on the project's three questions.
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
]

for i, question in enumerate(questions, start=1):
    print("=" * 60)
    print(f"Query {i}: {question}")
    print("=" * 60)
    run = gaming_industry_agent.invoke(question, session_id=f"q{i}")
    print(f"\nResponse:\n{extract_final_text(run)}\n")


Query 1: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Response:
Mortal Kombat X was not released for the PlayStation 5. It was originally released on April 14, 2015, for PlayStation 4, Xbox One, and Microsoft Windows. An upgraded version, Mortal Kombat XL, was released later for the same platforms, but there has been no specific release for the PlayStation 5 (source: [Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)). 

While Mortal Kombat X can be played on PlayStation 5 through backward compatibility with PlayStation 4 games, it was not specifically developed or released as a native PlayStation 5 title.



### (Optional) Advanced